# 32 — Trade Charts (Plotly, MT5-style)

For every **traded** position in the per-symbol broker logs (`logs/<SYMBOL>-YYYY-MM-DD.json`), draw a candlestick chart with:

* MT5-style **entry / exit arrows**
  * BUY  → blue arrow-up at entry, red arrow-down at exit
  * SELL → red arrow-down at entry, blue arrow-up at exit
  * entry & exit connected by a dashed line
* SL / TP horizontal lines
* previous **2 trades** of the same symbol overlaid as context
* **all signals** for this symbol that fall inside the chart window (faded outline triangles)
* **strategy indicators** (`ema-h1trend`, same definitions as `mt5/multi_symbol_bot/strategy.py`):
    * **EMA20** + **Bollinger Bands (20, 2σ)** overlaid on price
    * **RSI(14)** subplot with the 35 / 65 reaction bands
    * **ADX(14)** subplot with the 20 trend-strength floor
    * snapshot box (top-left) with the indicator values *as logged at the moment the signal fired*: `trend_dir`, `atr`, `adx`, `rsi`, `h1_rsi`, `risk`

Charts are rendered **7 at a time** — call `render_page(N)` to step through pages.

> Source data
> * Trades / signals: `logs/<SYMBOL>-YYYY-MM-DD.json` (one JSON event per line)
> * Candles:          `notebooks/data/<SYMBOL>/M5/ohlcv.csv`
> * Timestamps in the logs are tagged `+00:00` but actually reflect broker (Nicosia) time — we treat them as naive local broker time for plotting.

In [19]:
from __future__ import annotations

import json
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

In [20]:
# ---- paths ------------------------------------------------------------------
NB_DIR   = Path.cwd()
REPO     = NB_DIR.parent
LOG_DIR  = REPO / 'logs'
DATA_DIR = NB_DIR / 'data'

# render config
CHARTS_PER_PAGE   = 7         # how many trade charts per page
BARS_BEFORE_ENTRY = 60        # candles to show before entry
BARS_AFTER_EXIT   = 20        # candles to show after exit (open trades fall back to BARS_AFTER_ENTRY)
BARS_AFTER_ENTRY  = 40        # used when trade is still open
PREV_TRADES_CTX   = 2         # overlay previous N trades of same symbol
SHOW_ALL_SIGNALS  = True      # if True ignore SIGNALS_CTX and draw every signal inside the chart window
SIGNALS_CTX       = 5         # only used when SHOW_ALL_SIGNALS=False
TIMEFRAME         = 'M5'

# indicator periods — must match mt5/multi_symbol_bot/strategy.py
EMA_FAST          = 20
BB_PERIOD         = 20
BB_STD            = 2.0
RSI_PERIOD        = 14
ATR_PERIOD        = 14
ADX_PERIOD        = 14
RSI_OS            = 35.0
RSI_OB            = 65.0
ADX_MIN           = 20.0      # trend-strength floor that the live strategy applies

COLOR_BUY  = '#1f77ff'        # blue
COLOR_SELL = '#ff2d2d'        # red
COLOR_EMA  = '#ff9800'        # orange
COLOR_BB   = '#9e9e9e'        # grey

print(f'LOG_DIR  : {LOG_DIR}')
print(f'DATA_DIR : {DATA_DIR}')

LOG_DIR  : d:\bot\ema-1d trend\ema-h1trend\logs
DATA_DIR : d:\bot\ema-1d trend\ema-h1trend\notebooks\data


## 1. Load events from per-symbol broker logs

In [21]:
WANTED_EVENTS = {'signal', 'market_order_placed', 'pending_order_placed',
                 'position_closed_detected', 'position_opened',
                 'order_filled', 'order_cancelled'}

def _norm_symbol(sym: str) -> str:
    """Strip broker suffixes like `XAUUSD.i` -> `XAUUSD`."""
    if not isinstance(sym, str):
        return sym
    return sym.split('.')[0]

def load_symbol_log(path: Path) -> pd.DataFrame:
    rows = []
    with path.open('r', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if obj.get('event') not in WANTED_EVENTS:
                continue
            rows.append(obj)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    # broker time is the trader-relevant timestamp; fall back to ts
    if 'timeBroker' in df.columns:
        ts = df['timeBroker'].astype(str).str.replace(' BROKER', '', regex=False)
        df['ts_broker'] = pd.to_datetime(ts, errors='coerce')
    df['ts_utc'] = pd.to_datetime(df['ts'], errors='coerce', utc=True).dt.tz_convert(None)
    df['symbol_norm'] = df['symbol'].map(_norm_symbol) if 'symbol' in df.columns else None
    return df

log_files = sorted(p for p in LOG_DIR.glob('*-2*.json'))
print(f'Found {len(log_files)} per-symbol/day log files:')
for p in log_files:
    print('  ', p.name)

Found 7 per-symbol/day log files:
   CADCHF-2026-05-26.json
   EURCAD-2026-05-26.json
   EURUSD-2026-05-26.json
   GBPCAD-2026-05-26.json
   GBPUSD-2026-05-26.json
   USDMXN-2026-05-26.json
   XAUUSD-2026-05-26.json


In [22]:
all_events = pd.concat([load_symbol_log(p) for p in log_files], ignore_index=True)
print(all_events['event'].value_counts())
all_events.head()

event
signal                      20
market_order_placed          3
position_closed_detected     3
Name: count, dtype: int64


,ts,timeUTC,timeBroker,timeNY,event,symbol,direction,entry,sl,tp,bar_time,atr,risk,rr,trend_dir,adx,h1_rsi,rsi,ticket,side,volume,price,slippage_pts,ob_entry,profit,balance,equity,ts_broker,ts_utc,symbol_norm
0,2026-05-26T12:05:01.857195+00:00,2026-05-26 12:05:01 UTC,2026-05-26 15:05:01 BROKER,2026-05-26 08:05:01 NY,signal,EURCAD.i,BUY,1.60699,1.60633,1.60831,2026-05-26 15:00:00,0.00032,0.00066,2.0,1.0,19.4,57.3,58.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-26 15:05:01,2026-05-26 12:05:01.857195,EURCAD
1,2026-05-26T12:05:02.170211+00:00,2026-05-26 12:05:02 UTC,2026-05-26 15:05:02 BROKER,2026-05-26 08:05:02 NY,market_order_placed,EURCAD.i,NaN,NaN,1.60633,1.60876,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,39388277.0,buy,0.04,1.60714,15.0,1.60699,NaN,NaN,NaN,2026-05-26 15:05:02,2026-05-26 12:05:02.170211,EURCAD
2,2026-05-26T12:10:01.816794+00:00,2026-05-26 12:10:01 UTC,2026-05-26 15:10:01 BROKER,2026-05-26 08:10:01 NY,signal,EURCAD.i,BUY,1.60684,1.60632,1.60787,2026-05-26 15:05:00,0.00036,0.00052,2.0,1.0,18.0,55.8,53.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-26 15:10:01,2026-05-26 12:10:01.816794,EURCAD
3,2026-05-26T12:15:01.624177+00:00,2026-05-26 12:15:01 UTC,2026-05-26 15:15:01 BROKER,2026-05-26 08:15:01 NY,signal,EURCAD.i,BUY,1.60683,1.60632,1.60784,2026-05-26 15:10:00,0.00037,0.00051,2.0,1.0,16.8,55.7,53.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-26 15:15:01,2026-05-26 12:15:01.624177,EURCAD
4,2026-05-26T12:20:01.699161+00:00,2026-05-26 12:20:01 UTC,2026-05-26 15:20:01 BROKER,2026-05-26 08:20:01 NY,signal,EURCAD.i,BUY,1.60681,1.60632,1.60778,2026-05-26 15:15:00,0.00035,0.00049,2.0,1.0,15.6,55.4,52.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-05-26 15:20:01,2026-05-26 12:20:01.699161,EURCAD


## 2. Pair entry → exit events into trades

A *trade* = `market_order_placed` (or `order_filled`) matched to the next
`position_closed_detected` with the same `ticket`. Trades still open at the end
of the log keep `exit_ts = NaT`.

In [23]:
@dataclass
class Trade:
    symbol:     str
    ticket:     int
    side:       str           # 'buy' / 'sell'
    entry_ts:   pd.Timestamp
    entry_px:   float
    sl:         Optional[float]
    tp:         Optional[float]
    volume:     Optional[float]
    exit_ts:    Optional[pd.Timestamp] = None
    exit_px:    Optional[float]        = None
    profit:     Optional[float]        = None
    signal:     Optional[dict]         = None  # paired pre-trade signal row

OPEN_EVENTS = {'market_order_placed', 'order_filled', 'position_opened'}

def build_trades(events: pd.DataFrame) -> list[Trade]:
    if events.empty:
        return []
    events = events.sort_values('ts_utc').reset_index(drop=True)
    opens  = events[events['event'].isin(OPEN_EVENTS)].copy()
    closes = events[events['event'] == 'position_closed_detected'].copy()
    signals= events[events['event'] == 'signal'].copy()

    closes_by_ticket = {int(t): df for t, df in closes.groupby('ticket')}

    trades: list[Trade] = []
    for _, row in opens.iterrows():
        tkt = row.get('ticket')
        if pd.isna(tkt):
            continue
        tkt = int(tkt)
        entry_ts = row.get('ts_broker') or row['ts_utc']

        cl = closes_by_ticket.get(tkt)
        exit_row = None
        if cl is not None and not cl.empty:
            cl_after = cl[cl['ts_utc'] >= row['ts_utc']]
            if not cl_after.empty:
                exit_row = cl_after.iloc[0]

        # find the signal row that triggered this entry: same symbol, <= 10 min before
        sig_match = signals[(signals['symbol_norm'] == row['symbol_norm']) &
                            (signals['ts_utc'] <= row['ts_utc']) &
                            (signals['ts_utc'] >= row['ts_utc'] - pd.Timedelta(minutes=10))]
        sig_dict = sig_match.iloc[-1].to_dict() if not sig_match.empty else None

        trades.append(Trade(
            symbol   = row['symbol_norm'],
            ticket   = tkt,
            side     = str(row.get('side', '')).lower(),
            entry_ts = entry_ts,
            entry_px = float(row.get('price', np.nan)),
            sl       = float(row['sl']) if pd.notna(row.get('sl')) else None,
            tp       = float(row['tp']) if pd.notna(row.get('tp')) else None,
            volume   = float(row['volume']) if pd.notna(row.get('volume')) else None,
            exit_ts  = (exit_row.get('ts_broker') or exit_row['ts_utc']) if exit_row is not None else None,
            exit_px  = None,  # broker log records profit but not the closing price; we'll infer from candles
            profit   = float(exit_row['profit']) if exit_row is not None and pd.notna(exit_row.get('profit')) else None,
            signal   = sig_dict,
        ))
    return trades

trades = build_trades(all_events)
print(f'Built {len(trades)} trades.')
for t in trades:
    print(f'  {t.symbol:8s} {t.side:4s}  entry {t.entry_ts}  exit {t.exit_ts}  ticket {t.ticket}  P&L {t.profit}')

Built 3 trades.
  XAUUSD   sell  entry 2026-05-26 14:20:01  exit 2026-05-26 14:40:00  ticket 39385523  P&L 0.0
  EURCAD   buy   entry 2026-05-26 15:05:02  exit 2026-05-26 16:30:02  ticket 39388277  P&L 0.0
  GBPCAD   buy   entry 2026-05-26 15:25:01  exit 2026-05-26 16:35:01  ticket 39389259  P&L 0.0


## 3. Load M5 candles + compute strategy indicators

Definitions are intentionally identical to `mt5/multi_symbol_bot/strategy.py`:
* EMA20 = `close.ewm(span=20)`
* BB(20, 2σ) = `close.rolling(20).mean() ± 2 * close.rolling(20).std()`
* RSI(14) = Wilder smoothing
* ATR(14) = `TR.ewm(span=14)`
* ADX(14) = standard Wilder ADX

If these ever diverge from the live bot, the chart will lie — keep them in sync.

In [24]:
_candle_cache: dict[str, pd.DataFrame] = {}

def _ema(s: pd.Series, n: int) -> pd.Series:
    return s.ewm(span=n, adjust=False).mean()

def _rsi(close: pd.Series, n: int = 14) -> pd.Series:
    diff = close.diff()
    up = diff.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean()
    dn = (-diff.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean()
    rs = up / dn.replace(0, np.nan)
    return (100 - 100 / (1 + rs)).fillna(50)

def _atr(df: pd.DataFrame, n: int = 14) -> pd.Series:
    tr = pd.concat([
        df['high'] - df['low'],
        (df['high'] - df['close'].shift()).abs(),
        (df['low']  - df['close'].shift()).abs(),
    ], axis=1).max(axis=1)
    return tr.ewm(span=n, adjust=False).mean()

def _adx(df: pd.DataFrame, n: int = 14) -> pd.Series:
    up   =  df['high'].diff()
    down = -df['low'].diff()
    plus_dm  = pd.Series(np.where((up > down) & (up > 0), up, 0.0),   index=df.index)
    minus_dm = pd.Series(np.where((down > up) & (down > 0), down, 0.0), index=df.index)
    tr = pd.concat([
        df['high'] - df['low'],
        (df['high'] - df['close'].shift()).abs(),
        (df['low']  - df['close'].shift()).abs(),
    ], axis=1).max(axis=1)
    atr_w = tr.ewm(alpha=1 / n, adjust=False, min_periods=n).mean()
    plus_di  = 100 * plus_dm.ewm(alpha=1 / n, adjust=False, min_periods=n).mean()  / atr_w.replace(0, np.nan)
    minus_di = 100 * minus_dm.ewm(alpha=1 / n, adjust=False, min_periods=n).mean() / atr_w.replace(0, np.nan)
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)
    return dx.ewm(alpha=1 / n, adjust=False, min_periods=n).mean().fillna(0)

def _add_indicators(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['ema20'] = _ema(df['close'], EMA_FAST)
    mid         = df['close'].rolling(BB_PERIOD).mean()
    std         = df['close'].rolling(BB_PERIOD).std()
    df['bb_up'] = mid + BB_STD * std
    df['bb_lo'] = mid - BB_STD * std
    df['rsi']   = _rsi(df['close'], RSI_PERIOD)
    df['atr']   = _atr(df, ATR_PERIOD)
    df['adx']   = _adx(df, ADX_PERIOD)
    return df

def load_candles(symbol: str, timeframe: str = TIMEFRAME) -> pd.DataFrame:
    key = f'{symbol}/{timeframe}'
    if key in _candle_cache:
        return _candle_cache[key]
    path = DATA_DIR / symbol / timeframe / 'ohlcv.csv'
    if not path.exists():
        print(f'[warn] no candles at {path}')
        _candle_cache[key] = pd.DataFrame()
        return _candle_cache[key]
    df = pd.read_csv(path)
    df['time'] = pd.to_datetime(df['time'], utc=True, errors='coerce').dt.tz_convert(None)
    df = df.dropna(subset=['time']).sort_values('time').reset_index(drop=True)
    df = _add_indicators(df)
    _candle_cache[key] = df
    return df

for sym in {t.symbol for t in trades}:
    c = load_candles(sym)
    print(f'{sym}: {len(c):,} candles  ({c["time"].min()} .. {c["time"].max()})  '
          f'indicators: ema20, bb_up/lo, rsi, atr, adx')

XAUUSD: 100,000 candles  (2024-12-26 10:15:00 .. 2026-05-27 07:25:00)  indicators: ema20, bb_up/lo, rsi, atr, adx
GBPCAD: 100,000 candles  (2025-01-22 09:35:00 .. 2026-05-26 23:05:00)  indicators: ema20, bb_up/lo, rsi, atr, adx
EURCAD: 100,000 candles  (2025-01-22 09:45:00 .. 2026-05-26 23:05:00)  indicators: ema20, bb_up/lo, rsi, atr, adx


## 4. Plot helper — one trade, three subplots (price + RSI + ADX)

In [25]:
def _infer_exit_price(candles: pd.DataFrame, trade: Trade) -> Optional[float]:
    """If the broker log didn't capture the close price, take the candle close at exit_ts."""
    if trade.exit_ts is None or candles.empty:
        return None
    idx = candles['time'].searchsorted(trade.exit_ts)
    if idx >= len(candles):
        idx = len(candles) - 1
    return float(candles.iloc[idx]['close'])

def _window(candles: pd.DataFrame, trade: Trade) -> pd.DataFrame:
    if candles.empty:
        return candles
    start_idx = candles['time'].searchsorted(trade.entry_ts) - BARS_BEFORE_ENTRY
    if trade.exit_ts is not None:
        end_idx = candles['time'].searchsorted(trade.exit_ts) + BARS_AFTER_EXIT
    else:
        end_idx = candles['time'].searchsorted(trade.entry_ts) + BARS_AFTER_ENTRY
    start_idx = max(0, start_idx)
    end_idx   = min(len(candles), end_idx)
    return candles.iloc[start_idx:end_idx]

def _add_trade_markers(fig: go.Figure, trade: Trade, *, primary: bool, row=1, col=1):
    """MT5-style entry/exit arrows + dashed connector for a single trade.
    primary=False renders context trades with lower opacity and no SL/TP lines."""
    side = trade.side
    entry_color = COLOR_BUY if side == 'buy' else COLOR_SELL
    exit_color  = COLOR_SELL if side == 'buy' else COLOR_BUY
    entry_sym   = 'triangle-up'   if side == 'buy' else 'triangle-down'
    exit_sym    = 'triangle-down' if side == 'buy' else 'triangle-up'
    opacity     = 1.0 if primary else 0.4
    size_entry  = 16 if primary else 10
    size_exit   = 14 if primary else 9

    label = f"#{trade.ticket} {side.upper()}"
    fig.add_trace(go.Scatter(
        x=[trade.entry_ts], y=[trade.entry_px],
        mode='markers',
        marker=dict(symbol=entry_sym, color=entry_color, size=size_entry,
                    line=dict(color='black', width=1)),
        opacity=opacity,
        name=f'{label} entry',
        hovertemplate=(f'<b>{label} ENTRY</b><br>%{{x|%Y-%m-%d %H:%M}}<br>'
                       f'price=%{{y}}<extra></extra>'),
        showlegend=primary,
        legendgroup=label,
    ), row=row, col=col)

    if trade.exit_ts is not None and trade.exit_px is not None:
        fig.add_trace(go.Scatter(
            x=[trade.exit_ts], y=[trade.exit_px],
            mode='markers',
            marker=dict(symbol=exit_sym, color=exit_color, size=size_exit,
                        line=dict(color='black', width=1)),
            opacity=opacity,
            name=f'{label} exit',
            hovertemplate=(f'<b>{label} EXIT</b><br>%{{x|%Y-%m-%d %H:%M}}<br>'
                           f'price=%{{y}}<br>profit={trade.profit}<extra></extra>'),
            showlegend=False,
            legendgroup=label,
        ), row=row, col=col)
        fig.add_trace(go.Scatter(
            x=[trade.entry_ts, trade.exit_ts],
            y=[trade.entry_px, trade.exit_px],
            mode='lines',
            line=dict(color=entry_color, width=1.2, dash='dash'),
            opacity=opacity * 0.9,
            name=f'{label} path',
            hoverinfo='skip',
            showlegend=False,
            legendgroup=label,
        ), row=row, col=col)

    if primary and (trade.sl is not None or trade.tp is not None):
        x_end = trade.exit_ts or trade.entry_ts + pd.Timedelta(minutes=5 * BARS_AFTER_ENTRY)
        if trade.sl is not None:
            fig.add_shape(type='line', x0=trade.entry_ts, x1=x_end,
                          y0=trade.sl, y1=trade.sl,
                          line=dict(color='#d62728', width=1, dash='dot'),
                          row=row, col=col)
            fig.add_annotation(x=x_end, y=trade.sl, text=f' SL {trade.sl}',
                               showarrow=False, xanchor='left',
                               font=dict(color='#d62728', size=10),
                               row=row, col=col)
        if trade.tp is not None:
            fig.add_shape(type='line', x0=trade.entry_ts, x1=x_end,
                          y0=trade.tp, y1=trade.tp,
                          line=dict(color='#2ca02c', width=1, dash='dot'),
                          row=row, col=col)
            fig.add_annotation(x=x_end, y=trade.tp, text=f' TP {trade.tp}',
                               showarrow=False, xanchor='left',
                               font=dict(color='#2ca02c', size=10),
                               row=row, col=col)

def _add_signal_markers(fig: go.Figure, signals: pd.DataFrame, row=1, col=1):
    if signals.empty:
        return
    for _, sig in signals.iterrows():
        direction = str(sig.get('direction', '')).upper()
        is_buy    = direction == 'BUY'
        color     = COLOR_BUY if is_buy else COLOR_SELL
        sym       = 'triangle-up-open' if is_buy else 'triangle-down-open'
        ts        = sig.get('ts_broker') or sig.get('ts_utc')
        px        = sig.get('entry')
        if pd.isna(ts) or pd.isna(px):
            continue
        fig.add_trace(go.Scatter(
            x=[ts], y=[px],
            mode='markers',
            marker=dict(symbol=sym, color=color, size=11,
                        line=dict(color=color, width=1.5)),
            opacity=0.6,
            name=f'signal {direction}',
            hovertemplate=('<b>SIGNAL %{text}</b><br>%{x|%Y-%m-%d %H:%M}<br>'
                           'entry=%{y}<extra></extra>'),
            text=[direction],
            showlegend=False,
        ), row=row, col=col)

def _signal_snapshot(trade: Trade) -> str:
    """Build the multi-line indicator-at-signal-time annotation."""
    sig = trade.signal or {}
    def _f(key, fmt='{:.4f}'):
        v = sig.get(key)
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return '—'
        try:
            return fmt.format(float(v))
        except Exception:
            return str(v)
    trend = sig.get('trend_dir')
    trend_lbl = 'UP' if trend == 1 else ('DN' if trend == -1 else 'flat')
    return (
        f"<b>{trade.symbol} signal snapshot</b><br>"
        f"trend_dir : {trend_lbl}<br>"
        f"adx       : {_f('adx', '{:.1f}')}<br>"
        f"rsi       : {_f('rsi', '{:.1f}')}<br>"
        f"h1_rsi    : {_f('h1_rsi', '{:.1f}')}<br>"
        f"atr       : {_f('atr')}<br>"
        f"risk      : {_f('risk')}"
    )

def plot_trade(trade: Trade,
               all_trades: list[Trade],
               all_signals: pd.DataFrame) -> go.Figure:
    candles = load_candles(trade.symbol)
    win     = _window(candles, trade)

    if trade.exit_px is None:
        trade.exit_px = _infer_exit_price(candles, trade)

    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True,
        row_heights=[0.62, 0.19, 0.19],
        vertical_spacing=0.025,
        subplot_titles=('', f'RSI({RSI_PERIOD})', f'ADX({ADX_PERIOD})'),
    )

    if not win.empty:
        # — price subplot (row 1)
        fig.add_trace(go.Candlestick(
            x=win['time'], open=win['open'], high=win['high'],
            low=win['low'], close=win['close'],
            name=trade.symbol,
            increasing_line_color='#26a69a',
            decreasing_line_color='#ef5350',
            showlegend=False,
        ), row=1, col=1)
        fig.add_trace(go.Scatter(x=win['time'], y=win['ema20'],
                                 mode='lines', name=f'EMA{EMA_FAST}',
                                 line=dict(color=COLOR_EMA, width=1.4),
                                 hovertemplate='ema20=%{y:.5f}<extra></extra>'),
                      row=1, col=1)
        fig.add_trace(go.Scatter(x=win['time'], y=win['bb_up'],
                                 mode='lines', name=f'BB upper',
                                 line=dict(color=COLOR_BB, width=1, dash='dot'),
                                 hovertemplate='bb_up=%{y:.5f}<extra></extra>'),
                      row=1, col=1)
        fig.add_trace(go.Scatter(x=win['time'], y=win['bb_lo'],
                                 mode='lines', name='BB lower',
                                 line=dict(color=COLOR_BB, width=1, dash='dot'),
                                 fill='tonexty', fillcolor='rgba(158,158,158,0.06)',
                                 hovertemplate='bb_lo=%{y:.5f}<extra></extra>'),
                      row=1, col=1)

        # — RSI subplot (row 2)
        fig.add_trace(go.Scatter(x=win['time'], y=win['rsi'],
                                 mode='lines', name='RSI',
                                 line=dict(color='#7e57c2', width=1.2),
                                 showlegend=False,
                                 hovertemplate='rsi=%{y:.1f}<extra></extra>'),
                      row=2, col=1)
        for lvl, col_ in ((RSI_OB, '#ef5350'), (50, '#bdbdbd'), (RSI_OS, '#26a69a')):
            fig.add_hline(y=lvl, line=dict(color=col_, width=0.8, dash='dot'),
                          row=2, col=1)

        # — ADX subplot (row 3)
        fig.add_trace(go.Scatter(x=win['time'], y=win['adx'],
                                 mode='lines', name='ADX',
                                 line=dict(color='#ff7043', width=1.2),
                                 showlegend=False,
                                 hovertemplate='adx=%{y:.1f}<extra></extra>'),
                      row=3, col=1)
        fig.add_hline(y=ADX_MIN, line=dict(color='#bdbdbd', width=0.8, dash='dot'),
                      row=3, col=1)

    # previous N trades on same symbol — context overlay (price subplot only)
    prev_trades = [t for t in all_trades if t.symbol == trade.symbol
                   and t.entry_ts < trade.entry_ts][-PREV_TRADES_CTX:]
    for pt in prev_trades:
        if pt.exit_px is None:
            pt.exit_px = _infer_exit_price(candles, pt)
        _add_trade_markers(fig, pt, primary=False, row=1, col=1)

    # signals — all that fall inside the chart window (or last N if SHOW_ALL_SIGNALS=False)
    sig_same = all_signals[all_signals['symbol_norm'] == trade.symbol].copy()
    if not win.empty:
        win_lo, win_hi = win['time'].iloc[0], win['time'].iloc[-1]
        sig_ts = sig_same['ts_broker'].where(sig_same['ts_broker'].notna(),
                                              sig_same['ts_utc'])
        if SHOW_ALL_SIGNALS:
            sig_window = sig_same[(sig_ts >= win_lo) & (sig_ts <= win_hi)]
        else:
            sig_window = (sig_same[sig_ts <= (trade.exit_ts or trade.entry_ts)]
                          .tail(SIGNALS_CTX))
        _add_signal_markers(fig, sig_window, row=1, col=1)

    # primary trade markers last so they sit on top
    _add_trade_markers(fig, trade, primary=True, row=1, col=1)

    # indicator-snapshot box (top-left of the price subplot)
    fig.add_annotation(
        xref='paper', yref='paper', x=0.005, y=0.98,
        xanchor='left', yanchor='top',
        text=_signal_snapshot(trade),
        showarrow=False, align='left',
        bordercolor='#999', borderwidth=1, borderpad=4,
        bgcolor='rgba(255,255,255,0.85)',
        font=dict(size=10, family='monospace'),
    )

    if trade.exit_ts is not None:
        title = (f'{trade.symbol} #{trade.ticket}  {trade.side.upper()}  '
                 f'@ {trade.entry_ts:%Y-%m-%d %H:%M} → '
                 f'{trade.exit_ts:%Y-%m-%d %H:%M}  P&L={trade.profit}')
    else:
        title = (f'{trade.symbol} #{trade.ticket}  {trade.side.upper()}  '
                 f'@ {trade.entry_ts:%Y-%m-%d %H:%M}  (open)')

    fig.update_layout(
        title=title,
        height=720,
        margin=dict(l=40, r=120, t=60, b=30),
        template='plotly_white',
        hovermode='x unified',
        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                    xanchor='right', x=1),
    )
    fig.update_xaxes(rangeslider_visible=False, showspikes=True,
                     spikemode='across', row=1, col=1)
    fig.update_xaxes(showspikes=True, row=2, col=1)
    fig.update_xaxes(showspikes=True, row=3, col=1)
    fig.update_yaxes(showspikes=True, row=1, col=1)
    fig.update_yaxes(range=[0, 100], row=2, col=1)
    fig.update_yaxes(range=[0, 60],  row=3, col=1)
    return fig

## 5. Render — 7 charts per page

In [26]:
all_signals_df = all_events[all_events['event'] == 'signal'].copy()

def render_page(page: int = 0, charts_per_page: int = CHARTS_PER_PAGE):
    total = len(trades)
    pages = max(1, (total + charts_per_page - 1) // charts_per_page)
    page  = max(0, min(page, pages - 1))
    lo, hi = page * charts_per_page, min(total, (page + 1) * charts_per_page)
    print(f'Page {page + 1} / {pages}   (trades {lo + 1}..{hi} of {total})')
    for i in range(lo, hi):
        fig = plot_trade(trades[i], trades, all_signals_df)
        fig.show()

# default: first page
render_page(0)

Page 1 / 1   (trades 1..3 of 3)


In [27]:
# next page(s) — bump the number to step through
render_page(1)

Page 1 / 1   (trades 1..3 of 3)


### Tips
* Re-run `render_page(N)` with different `N` to flip pages.
* Tunables in cell 2: `CHARTS_PER_PAGE`, `BARS_BEFORE_ENTRY`, `BARS_AFTER_EXIT`, `PREV_TRADES_CTX`, `SHOW_ALL_SIGNALS` / `SIGNALS_CTX`.
* Indicator periods (`EMA_FAST`, `BB_PERIOD`, `RSI_PERIOD`, `ATR_PERIOD`, `ADX_PERIOD`) must match `mt5/multi_symbol_bot/strategy.py`. If you tune the live bot, update them here too.
* Re-run sections 1–2 to pick up newly-arrived daily log files.